# LexAI — Notebook 02: Document Processing

Demonstrates the multimodal document processing pipeline:
- **PDF extraction** with PyMuPDF
- **Named Entity Recognition** with spaCy
- **Legal charge identification** (IPC/CrPC section detection)
- **LLM document summaries** via Groq

Prerequisites: Run `scripts/generate_sample_cases.py` to create test documents.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

CASES_DIR = pathlib.Path('../data/cases/sample_cases')
print('Sample cases:', list(CASES_DIR.iterdir()) if CASES_DIR.exists() else 'Run generate_sample_cases.py first')

## 1. Initialize Document Processor

In [ ]:
from src.document_processor import MultimodalDocumentProcessor

processor = MultimodalDocumentProcessor()
print('Supported file types:', list(processor.SUPPORTED_TYPES.keys()))

## 2. Process a PDF Document

In [ ]:
pdf_path = CASES_DIR / 'case_001' / 'sale_deed.pdf'

if not pdf_path.exists():
    print('Sample PDF not found. Run: python -X utf8 scripts/generate_sample_cases.py')
else:
    result = processor.process_document(str(pdf_path))
    print('Document type:', result.get('doc_type'))
    print('File name:', result.get('file_name'))
    print('Text length:', len(result.get('text', '')))
    print('\nFirst 500 characters of extracted text:')
    print(result.get('text', '')[:500])

## 3. Named Entity Recognition

In [ ]:
sample_text = """
In the matter of Rajesh Kumar vs. Priya Sharma, FIR No. 145/2024 was registered
at Andheri Police Station, Mumbai on 15th March 2024 under IPC Section 420
and IPC Section 468. The accused defrauded the complainant of Rs. 45,00,000
through fake investment scheme. The Sessions Court, Mumbai is hearing this matter.
The next hearing is scheduled for 20th June 2024.
"""

entities = processor.extract_entities(sample_text)

print('=== Extracted Entities ===')
for entity_type, values in entities.items():
    if values:
        print(f'\n{entity_type.upper()}:')
        for v in values:
            print(f'  - {v}')

## 4. Legal Charge Identification

In [ ]:
charges = processor.identify_legal_charges(sample_text)

print('Identified IPC Sections:')
for charge in charges:
    print(f'  IPC Section {charge}')

## 5. Regex Pattern Testing

In [ ]:
import re

test_text = """
The accused committed offences under IPC 302, Sec 420, Section 498A and u/s 376.
The amount involved was Rs. 12,50,000 and INR 3,00,000.
FIR No. CR-2024-001 was filed on 01/03/2024 and 15-March-2024.
Case No. WP/2024/1234 is pending before the High Court.
"""

print('IPC matches:', re.findall(processor.IPC_PATTERN, test_text, re.IGNORECASE))
print('Money matches:', re.findall(processor.MONEY_PATTERN, test_text, re.IGNORECASE))
print('Date matches:', re.findall(processor.DATE_PATTERN, test_text))
print('Case No. matches:', re.findall(processor.CASE_NUM_PATTERN, test_text, re.IGNORECASE))

## 6. Process All Case 002 Documents

In [ ]:
case002_dir = CASES_DIR / 'case_002'

if case002_dir.exists():
    results = []
    for f in sorted(case002_dir.iterdir()):
        if f.suffix.lower() in processor.SUPPORTED_TYPES:
            r = processor.process_document(str(f))
            results.append({
                'file': f.name,
                'type': r.get('doc_type'),
                'text_len': len(r.get('text', '')),
                'entities': sum(len(v) for v in r.get('entities', {}).values()),
                'charges': r.get('legal_charges', [])
            })
    
    import pandas as pd
    df = pd.DataFrame(results)
    print(df.to_string(index=False))
else:
    print('Run generate_sample_cases.py first')

## Summary

The document processor handles:
- **PDF/DOCX/TXT** → text extraction
- **Regex patterns** → IPC sections, dates, monetary amounts, case numbers
- **spaCy NER** → persons, organisations, locations
- **LLM summaries** → via Groq (mocked when no API key)

Next: See Notebook 03 for how these documents are indexed in the RAG pipeline.